In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('D:\Dev\Internal-Tank-Registration\source')

import pickle
import os
import gc
from copy import deepcopy
import pandas as pd
import numpy as np
from glob import glob
from pathlib import Path
import matplotlib.pyplot as plt
from multiprocessing import Pool
import open3d as o3d
import time

from cylinder_alignment.bundle_station import BundleStation
from cylinder_extraction.station import Station
from cylinder_alignment.station_alignment import StationAlignment
from cylinder_alignment.utils import extract_target_cylinder
from utils.write_output import write_array2txt
from utils.math_methods import calc_overlap_curve

import plotly.express as px

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


5 mins
5


In [2]:
scan_field_names = [
    ('JB-569 120ft',8)
]
execuated_time = {}
for field, pack_num in scan_field_names:    
    time_begin = time.time()
    try:
        print('===========================')
        print(field)
        data_dir = Path(f'D:\REGISTRATION DATA\Internal - No FR/{field}')
        # READ EXTRACTED CYLINDERS DATA
        bundle = BundleStation(data_dir=data_dir)
        summary_df = bundle.read_extracted_cylinder_data(file_types=['e57'])
        # ############################
        # px.scatter(
        #     summary_df[summary_df.radius < 50], y='radius', x='station', 
        #     color='station', hover_data=summary_df.columns
        # )
        # PAIRWISE STATION ALIGNMENT
        bundle.align_pairwise_station()
        time_pairwise = time.time()
        # Save
        bundle.remove_points()
        with open(os.path.join(data_dir, 'rough_alignment.pkl'), 'wb') as f:
            pickle.dump(bundle, f)
        # load
        with open(os.path.join(data_dir, 'rough_alignment.pkl'), 'rb') as f:
            bundle = pickle.load(f)
        # uncertain stations
        # uncertain_stations = bundle.find_uncertain_stations()
        # if len(uncertain_stations) > 0:
        #     print('uncertain stations')
        #     print(uncertain_stations)
        #     raise ValueError('Uncertain stations exist')
        # bring all stations to a global coordinate
        bundle.align_all_stations_by_axis()
        # CLUSTER TANKS
        bundle.cluster_cylinder()
        time_cluster = time.time()
        # save
        bundle.remove_points()
        with open(os.path.join(data_dir, 'globalization_center_phase_1.pkl'), 'wb') as f:
            pickle.dump(bundle, f)
        with open(os.path.join(data_dir, 'globalization_center_phase_1.pkl'), 'rb') as f:
            bundle = pickle.load(f)
        # VISUAL INSPECTION
        # fig = bundle.draw_cross_section()
        # fig.show()
        
        # # TARGET TANK
        # bundle.identify_target_tank()
        # print('target tank', bundle.target_tank_id)
        # # bundle.set_target_tank(tank_id=3)
        # # Put all fragments of the target tank in one place. Include points at their foot
        # bundle.write_target_tank(
        #     write_dir=bundle.data_dir / 'target_tank' / 'local' / 'ground',
        #     load_data=True, extend_ground=True, transform=False)
        # # update location of all the segments of the target tank: load original ground-extended data
        # for c in bundle.target_tank:
        #     c.update_filepath(new_dir=bundle.data_dir / 'target_tank' / 'local' / 'ground', num_keep_level=1, file_name=f'{Path(c.ptc_filepath).stem}.pcd')
        #     print(c.ptc_filepath)

        # # alignment transformation matrices
        # bundle.calc_align_matrix(f'{bundle.target_tank_id}')
        # # save
        # bundle.remove_points()
        # with open(os.path.join(data_dir, 'globalization_axis_phase_1.pkl'), 'wb') as f:
        #     pickle.dump(bundle, f)
        # with open(os.path.join(data_dir, 'globalization_axis_phase_1.pkl'), 'rb') as f:
        #     bundle = pickle.load(f)

        # # write the transformed segments
        # bundle.write_target_tank(
        #     write_dir=bundle.data_dir / 'target_tank' / 'global' / 'ground', 
        #     load_data=True, transform=True, extend_ground=False, num_process=5)

        # # THIN SLICE
        # bundle.cut_horizontal(
        #     mode='station_cylinders', z_cut=3, thickness=0.01, 
        #     write_dir=bundle.data_dir / 'target_tank' / 'global' / 'thin_slice')
        # bundle.cut_horizontal(
        #     mode='target_tank', z_cut=3, thickness=0.01,
        #     write_dir=bundle.data_dir / 'target_tank' / 'global' / 'thin_slice')
    except Exception as exc:
        print(f'===== FAILED HANDLING SCAN {field} =====')
        print(exc)
    time_end = time.time()
    lasted = time_end - time_begin
    print(f'processing took {lasted} seconds')
    execuated_time[field] = lasted
    # del bundle
    # gc.collect()
        

JB-569 120ft
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\Tk12002_Scan_univar12001Scan_001-output\Tk12002_Scan_univar12001Scan_001.pkl
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\Tk12002_Scan_univar12001Scan_002-output\Tk12002_Scan_univar12001Scan_002.pkl
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\Tk12002_Scan_univar12001Scan_003-output\Tk12002_Scan_univar12001Scan_003.pkl
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\Tk12002_Scan_univar12001Scan_004-output\Tk12002_Scan_univar12001Scan_004.pkl
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\Tk12002_Scan_univar12001Scan_005-output\Tk12002_Scan_univar12001Scan_005.pkl
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\Tk12002_Scan_univar12001Scan_006-output\Tk12002_Scan_univar12001Scan_006.pkl
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\Tk12002_Scan_univar12001Scan_007-output\Tk12002_Scan_univar12001Scan_007.pkl
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\Tk12002_Scan_univar12001Scan_008-output\

In [31]:
with open(os.path.join(data_dir, 'globalization_center_phase_1.pkl'), 'rb') as f:
    bundle = pickle.load(f)

In [32]:
# the artificial cylinders do not have 'station' field, which will cause errors for following code
# a quick fix
for s in bundle.stations:
    for c in s.refined_cylinders:
        c.station = s

In [33]:
# fig = bundle.draw_cross_section()
# fig.show()

In [34]:
for cyl in bundle.tanks_info['tanks']['0']:
    if cyl.radius > 1:
        print(cyl)

{'station': {'name': 'Tk12002_Scan_univar12001Scan_001'}, 'radius': 18.293518, 'axis': "{'position': array([-1.4080248e+01, -4.0647922e+00, -1.0858186e-02], dtype=float32), 'vector': array([ 1.6768284e-04, -3.4004339e-04,  9.9999994e-01], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\Internal - No FR\\JB-569 120ft\\Tk12002_Scan_univar12001Scan_001-output\\Tk12002_Scan_univar12001Scan_001 cluster_1.pcd'}
{'station': {'name': 'Tk12002_Scan_univar12001Scan_002'}, 'radius': 18.293102, 'axis': "{'position': array([-12.744842  ,  -2.6274967 ,  -0.01339891], dtype=float32), 'vector': array([-9.0702856e-04, -4.5704140e-04,  9.9999946e-01], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\Internal - No FR\\JB-569 120ft\\Tk12002_Scan_univar12001Scan_002-output\\Tk12002_Scan_univar12001Scan_002 cluster_1.pcd'}
{'station': {'name': 'Tk12002_Scan_univar12001Scan_003'}, 'radius': 18.296423, 'axis': "{'position': array([-13.646965  ,  -1.2931564 ,  -0.01642564], dtype=float32), '

In [35]:
bundle.set_target_tank(tank_id=0)

In [16]:
bundle.write_target_tank(
    write_dir=bundle.data_dir / 'target_tank' / 'local' / 'no_ground',
    load_data=True, extend_ground=False, transform=False
)

In [36]:
for c in bundle.target_tank:
    c.update_filepath(new_dir=bundle.data_dir / 'target_tank' / 'local' / 'no_ground', num_keep_level=1, file_name=f'{Path(c.ptc_filepath).stem}.pcd')
    print(c.ptc_filepath)

D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\target_tank\local\no_ground\Tk12002_Scan_univar12001Scan_001 cluster_1.pcd
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\target_tank\local\no_ground\Tk12002_Scan_univar12001Scan_001 cluster_4.pcd
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\target_tank\local\no_ground\Tk12002_Scan_univar12001Scan_002 cluster_1.pcd
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\target_tank\local\no_ground\Tk12002_Scan_univar12001Scan_002 cluster_6.pcd
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\target_tank\local\no_ground\Tk12002_Scan_univar12001Scan_003 cluster_1.pcd
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\target_tank\local\no_ground\Tk12002_Scan_univar12001Scan_003 cluster_6.pcd
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\target_tank\local\no_ground\Tk12002_Scan_univar12001Scan_004 cluster_1.pcd
D:\REGISTRATION DATA\Internal - No FR\JB-569 120ft\target_tank\local\no_ground\Tk12002_Scan_univar12001Scan_004 cluste

In [37]:
valid_cyls = []
for cyl in bundle.target_tank:
    if cyl.radius > 1:
        valid_cyls.append(cyl)
bundle.target_tank = valid_cyls

In [38]:
# alignment transformation matrices
bundle.calc_align_matrix(f'{bundle.target_tank_id}')
# save
bundle.remove_points()
with open(os.path.join(data_dir, 'globalization_axis_phase_1.pkl'), 'wb') as f:
    pickle.dump(bundle, f)
with open(os.path.join(data_dir, 'globalization_axis_phase_1.pkl'), 'rb') as f:
    bundle = pickle.load(f)

# write the transformed segments
bundle.write_target_tank(
    write_dir=bundle.data_dir / 'target_tank' / 'global' / 'no_ground', 
    load_data=True, transform=True, extend_ground=False, num_process=5
)

calculating alignment matrix for tank 0
Tk12002_Scan_univar12001Scan_001 cluster 1
Tk12002_Scan_univar12001Scan_002 cluster 1
Tk12002_Scan_univar12001Scan_003 cluster 1
Tk12002_Scan_univar12001Scan_004 cluster 1
Tk12002_Scan_univar12001Scan_005 cluster 1
Tk12002_Scan_univar12001Scan_006 cluster 1
Tk12002_Scan_univar12001Scan_007 cluster 1
Tk12002_Scan_univar12001Scan_008 cluster 1
Tk12002_Scan_univar12001Scan_009 cluster 1
Tk12002_Scan_univar12001Scan_011 cluster 1
Tk12002_Scan_univar12001Scan_011 cluster 2
Tk12002_Scan_univar12001Scan_012 cluster 1
Tk12002_Scan_univar12001Scan_013 cluster 1
Tk12002_Scan_univar12001Scan_014 cluster 1
Tk12002_Scan_univar12001Scan_016 cluster 1


In [19]:
execuated_time

{'JB-505 100ft EXT Trimble': 2652.029712677002}

In [2]:
data_dir = Path(f'D:/REGISTRATION DATA/External/JB-505 100ft EXT Trimble')

In [3]:
with open(os.path.join(data_dir, 'globalization_center_phase_1.pkl'), 'rb') as f:
    bundle = pickle.load(f)

In [22]:
a = np.random.rand(3)
a = np.sort(a)
a

array([0.26259395, 0.32570422, 0.80599012])

In [23]:
b = np.r_[0, a, 2*np.pi]

In [24]:
b

array([0.        , 0.26259395, 0.32570422, 0.80599012, 6.28318531])

In [27]:
np.sum(np.diff(b)) / (2*np.pi)

1.0

In [4]:
bundle.identify_target_tank()

span [0.12387674996013776, 0.39200815263957156, 0.502873791841946, 0.5919053703393555, 0.5262141485156824, 0.3577527339922416, 0.4301320760213696, 0.24500650995045892, 0.4593352369873045, 0.5269882434126241, 0.2492767008364698, 0.5284045009185527, 0.5554518819137297, 0.2598230979333286, 0.10016649098216446, 1.0, 0.22784227191411477, 0.6242515181131789, 0.09525991445317783]


15

In [5]:
bundle.write_target_tank(
    write_dir=bundle.data_dir / 'target_tank' / 'local' / 'ground',
    load_data=True, extend_ground=True, transform=False)
# update location of all the segments of the target tank: load original ground-extended data
for c in bundle.target_tank:
    c.update_filepath(new_dir=bundle.data_dir / 'target_tank' / 'local' / 'ground', num_keep_level=1, file_name=f'{Path(c.ptc_filepath).stem}.pcd')
    print(c.ptc_filepath)

# alignment transformation matrices
bundle.calc_align_matrix(f'{bundle.target_tank_id}')
# save
bundle.remove_points()
with open(os.path.join(data_dir, 'globalization_axis_phase_1.pkl'), 'wb') as f:
    pickle.dump(bundle, f)
with open(os.path.join(data_dir, 'globalization_axis_phase_1.pkl'), 'rb') as f:
    bundle = pickle.load(f)

# write the transformed segments
bundle.write_target_tank(
    write_dir=bundle.data_dir / 'target_tank' / 'global' / 'ground', 
    load_data=True, transform=True, extend_ground=False, num_process=5)

# THIN SLICE
bundle.cut_horizontal(
    mode='station_cylinders', z_cut=3, thickness=0.01, 
    write_dir=bundle.data_dir / 'target_tank' / 'global' / 'thin_slice')
bundle.cut_horizontal(
    mode='target_tank', z_cut=3, thickness=0.01,
    write_dir=bundle.data_dir / 'target_tank' / 'global' / 'thin_slice')

MemoryError: Unable to allocate 3.10 GiB for an array with shape (3, 138744384) and data type float64